In [4]:
# DNSC 6330 - Individual Homework 1: R → Python (COMPAS ProPublica Workflow)
# Author: Farid Samedli


import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.metrics import confusion_matrix

#LOAD DATA
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
raw_data = pd.read_csv(url)
print(f"Raw records: {len(raw_data):,}")

#PREPROCESSING 
cols = ['age', 'c_charge_degree', 'race', 'age_cat', 'score_text', 'sex',
        'priors_count', 'days_b_screening_arrest', 'decile_score',
        'is_recid', 'two_year_recid', 'c_jail_in', 'c_jail_out']

df = raw_data[cols].copy()

df = df[(df['days_b_screening_arrest'] >= -30) & (df['days_b_screening_arrest'] <= 30)]
df = df[df['is_recid'] != -1]
df = df[df['c_charge_degree'] != 'O']
df = df[df['score_text'] != 'N/A']

print(f"Rows after filtering: {len(df):,}")

#FACTOR ENGINEERING
df['crime_factor'] = df['c_charge_degree'].astype('category')
df['age_factor'] = pd.Categorical(df['age_cat'])
df['race_factor'] = pd.Categorical(df['race'])
df['gender_factor'] = pd.Categorical(df['sex'], categories=['Female', 'Male'])
df['score_factor'] = np.where(df['score_text'] != 'Low', 'HighScore', 'LowScore')
df['score_factor'] = pd.Categorical(df['score_factor'], categories=['LowScore', 'HighScore'])

#LOGISTIC REGRESSION 
formula = ("score_factor ~ C(gender_factor, Treatment('Male')) + "
           "C(age_factor, Treatment('25 - 45')) + "
           "C(race_factor, Treatment('Caucasian')) + "
           "priors_count + C(crime_factor) + two_year_recid")

model = smf.glm(formula=formula, data=df, family=sm.families.Binomial()).fit()
print(model.summary())

#FAIRNESS ANALYSIS 
df['compas_high_risk'] = (df['score_factor'] == 'HighScore').astype(int)

def error_rates(cm):
    tn, fp, fn, tp = cm.ravel()
    return {
        'FPR': fp / (fp + tn) if (fp + tn) > 0 else 0,
        'FNR': fn / (fn + tp) if (fn + tp) > 0 else 0
    }

print("\nError Rates by Race:")
for race, group in df.groupby('race'):
    cm = confusion_matrix(group['two_year_recid'], group['compas_high_risk'])
    rates = error_rates(cm)
    print(f"{race:15} FPR={rates['FPR']:.3f}  FNR={rates['FNR']:.3f}")

Raw records: 7,214
Rows after filtering: 6,172
                                   Generalized Linear Model Regression Results                                   
Dep. Variable:     ['score_factor[LowScore]', 'score_factor[HighScore]']   No. Observations:                 6172
Model:                                                               GLM   Df Residuals:                     6160
Model Family:                                                   Binomial   Df Model:                           11
Link Function:                                                     Logit   Scale:                          1.0000
Method:                                                             IRLS   Log-Likelihood:                -3084.2
Date:                                                   Mon, 30 Mar 2026   Deviance:                       6168.4
Time:                                                           15:30:44   Pearson chi2:                 6.07e+03
No. Iterations:                          